客户端

  ↓ HTTP / TCP

Uvicorn

  ↓ ASGI 消息

FastAPI

  ↓ 路由匹配、校验、业务逻辑
  
你的接口函数

- Uvicorn 是服务器程序，负责监听端口、接收网络请求。
- FastAPI 是 Web 框架，负责路由、参数校验、依赖注入和调用接口函数。
- ASGI 是它们之间约定的通信接口

> WSGI/ASGI是Web Server 与 Python Web Application 之间的接口规范

### WSGI 是什么？
全称`Web Server Gateway Interface`，服务于传统的同步 Python Web 应用
基本模型接近：
```
收到一个请求
→ 调用一次应用
→ 返回一个响应
```

#### WSGI 的限制
围绕传统的同步 HTTP 请求—响应周期设计，因此不适合原生表达：
- WebSocket 长连接
- 同一连接上的多次事件
- 异步接收和发送
- 大量长时间保持的连接

不表示 WSGI 不能处理并发。它可以借助：
- 多线程
- 多进程
- 多 Worker

> WSGI 的应用接口是同步的，不原生支持 ASGI 那种异步事件通信模型。

### ASGI 是什么？
全称`Asynchronous Server Gateway Interface`是面向现代异步 Python Web 应用的接口规范，也是 WSGI 的后继方案。
最简单的应用如下`async def application(scope, receive, send):`接受3个参数：
- scope：表示当前连接的基本信息，这是什么连接、访问什么路径，以及连接的基本元数据。
  - 请求对应一个 HTTP scope；对于 WebSocket，scope 的生命周期通常会持续到连接关闭。
- receive：异步接收客户端发来的事件
- send：异步向客户端发送事件

## Uvicorn 负责什么？
Uvicorn 是 ASGI Server。它主要负责：
- 监听 IP 和端口。
- 接受 TCP 连接。
- 解析 HTTP/WebSocket 协议。
- 将网络请求转换成 ASGI 的 scope 和事件。
- 调用 FastAPI 应用。
- 把 FastAPI 发出的 ASGI 响应事件转换成网络响应。
FastAPI 本身一般不会直接监听端口，所以仅仅写：
`app = FastAPI()`不会自动出现一个 HTTP 服务。
需要使用 Uvicorn：`uvicorn main:app --reload`

### FastAPI 为什么是 ASGI 应用？
`app = FastAPI()`相当于`await app(scope, receive, send)`

完整过程如下：
```
浏览器发送 GET /users
        ↓
Uvicorn 接收并解析 HTTP
        ↓
构造 ASGI scope、receive、send
        ↓
调用 FastAPI app
        ↓
FastAPI 匹配 GET /users
        ↓
执行依赖注入、参数校验和路由函数
        ↓
FastAPI 发送 ASGI 响应事件
        ↓
Uvicorn 转换成 HTTP 响应
        ↓
返回浏览器
```

ASGI 提供了异步运行的接口和基础，但不会自动把同步代码变成异步代码。

#### 面试模版
> Uvicorn 是负责监听端口和处理网络协议的 ASGI Server，FastAPI 是实现 ASGI 接口的 Web 框架，ASGI 是两者之间的标准。Uvicorn 把 HTTP 或 WebSocket 连接转换成 scope、receive 和 send，FastAPI 再完成路由匹配、参数校验、依赖注入和业务处理。